In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import weibull_min

def compute_distribution(
    csv_path,
    wdir_bin_width=30,
    wsp_bin_width=2,
    wsp_range=(0, 34),
    min_samples=10,
    fill_missing_directions=True,
    output_path="distribution_output.csv"
):
    

    # Load and preprocess
    df = pd.read_csv(csv_path, parse_dates=['timestamp'])
    df = df.dropna(subset=['wsp', 'wdir', 'TI'])
    df['wdir'] = df['wdir'] % 360

    wdir_edges = np.arange(0, 360 + wdir_bin_width, wdir_bin_width)
    wdir_centers = (wdir_edges[:-1] + wdir_edges[1:]) / 2
    df['wdir_bin'] = pd.cut(df['wdir'], bins=wdir_edges, labels=wdir_centers, right=False).astype(float)

    wsp_edges = np.arange(wsp_range[0], wsp_range[1] + wsp_bin_width, wsp_bin_width)
    wsp_centers = np.round((wsp_edges[:-1] + wsp_edges[1:]) / 2, 2)

    mean_TI = round(df['TI'].mean(), 2)
    total_count = len(df)

    output_rows = []
    previous_fit = None

    for wdir_bin in wdir_centers:
        wsp_data = df[df['wdir_bin'] == wdir_bin]['wsp']
        n = len(wsp_data)
        p_dir = n / total_count

        if n >= min_samples:
            c, loc, scale = weibull_min.fit(wsp_data, floc=0)
            previous_fit = (c, scale)
        elif fill_missing_directions and previous_fit is not None:
            c, scale = previous_fit
        else:
            continue  # skip this bin

        # Weibull CDF to get bin probabilities
        cdf_vals = weibull_min(c, loc=0, scale=scale).cdf(wsp_edges)
        cond_probs = np.diff(cdf_vals)  # p(wsp | wdir)

        # Joint probability: p(wsp, wdir) = p(wdir) * p(wsp | wdir)
        for wsp_center, p_wsp_given_dir in zip(wsp_centers, cond_probs):
            p_joint = p_dir * p_wsp_given_dir
            output_rows.append({
                "wsp": round(wsp_center, 2),
                "TI": mean_TI,
                "wdir": round(wdir_bin, 2),
                "probability": p_joint,
                "n": n  # optional: remove if not needed
            })

    dist_df = pd.DataFrame(output_rows)

    # Normalize total probability to 1
    dist_df['probability'] /= dist_df['probability'].sum()

    dist_df.to_csv(output_path, index=False)
    return dist_df

output_path="/groups/SUDOCO/Task23/sudoco_task2.3/distributions/test.csv"
result = compute_distribution(
    csv_path="/groups/SUDOCO/Task23/sudoco_task2.3/data/timeseries/HKNB_timeseries.csv",
    wdir_bin_width=5,
    wsp_bin_width=1,
    wsp_range=(0, 34),
    fill_missing_directions=True,
    output_path=output_path
)


In [ ]:
# The same as the preboius but calcualting the mean TI per bin

import pandas as pd
import numpy as np
from scipy.stats import weibull_min

def compute_distribution(
    csv_path,
    wdir_bin_width=30,
    wsp_bin_width=2,
    wsp_range=(0, 34),
    min_samples=10,
    fill_missing_directions=True,
    output_path="distribution_output.csv"
):
    # Load and preprocess
    df = pd.read_csv(csv_path, parse_dates=['timestamp'])
    df = df.dropna(subset=['wsp', 'wdir', 'TI'])
    df['wdir'] = df['wdir'] % 360

    wdir_edges = np.arange(0, 360 + wdir_bin_width, wdir_bin_width)
    wdir_centers = (wdir_edges[:-1] + wdir_edges[1:]) / 2
    df['wdir_bin'] = pd.cut(df['wdir'], bins=wdir_edges, labels=wdir_centers, right=False).astype(float)

    wsp_edges = np.arange(wsp_range[0], wsp_range[1] + wsp_bin_width, wsp_bin_width)
    wsp_centers = np.round((wsp_edges[:-1] + wsp_edges[1:]) / 2, 2)

    total_count = len(df)
    mean_TI = df['TI'].mean()  

    output_rows = []
    previous_fit = None

    for wdir_bin in wdir_centers:
        df_dir = df[df['wdir_bin'] == wdir_bin].copy()
        wsp_data = df_dir['wsp']
        n = len(wsp_data)
        p_dir = n / total_count

        if n >= min_samples:
            c, loc, scale = weibull_min.fit(wsp_data, floc=0)
            previous_fit = (c, scale)
        elif fill_missing_directions and previous_fit is not None:
            c, scale = previous_fit
        else:
            continue  # skip this bin

        # Weibull CDF to get bin probabilities
        cdf_vals = weibull_min(c, loc=0, scale=scale).cdf(wsp_edges)
        cond_probs = np.diff(cdf_vals)  # p(wsp | wdir)

        # Loop over wind speed bins and compute mean TI
        df_dir['wsp_bin'] = pd.cut(df_dir['wsp'], bins=wsp_edges, labels=wsp_centers, right=False).astype(float)
        ti_by_bin = df_dir.groupby('wsp_bin', observed=True)['TI'].mean()
        


        for wsp_center, p_wsp_given_dir in zip(wsp_centers, cond_probs):
            p_joint = p_dir * p_wsp_given_dir
            mean_ti = ti_by_bin.get(wsp_center, np.nan)
            if np.isnan(mean_ti):
                mean_ti = mean_TI  # fallback to site mean

            output_rows.append({
                "wsp": round(wsp_center, 2),
                "wdir": round(wdir_bin, 2),
                "TI": round(mean_ti, 4) if pd.notna(mean_ti) else np.nan,
                "probability": p_joint,
                "n": n
            })


    dist_df = pd.DataFrame(output_rows)

    # Normalize total probability to 1
    dist_df['probability'] /= dist_df['probability'].sum()

    dist_df.to_csv(output_path, index=False)
    return dist_df
output_path = "/groups/SUDOCO/Task23/sudoco_task2.3/distributions/test.csv"
result = compute_distribution(
    csv_path="/groups/SUDOCO/Task23/sudoco_task2.3/data/timeseries/HKNB_timeseries_full_filled_small_gaps_only.csv",
    wdir_bin_width=3,
    wsp_bin_width=1,
    wsp_range=(0, 34),
    fill_missing_directions=True,
    output_path=output_path
)

In [7]:
# The same as the preboius but calcualting the mean TI per bin and also clipping to only operational conditions and renormalizing

import pandas as pd
import numpy as np
from scipy.stats import weibull_min

def compute_distribution(
    csv_path,
    wdir_bin_width=30,
    wsp_bin_width=2,
    wsp_range=(0, 34),
    min_samples=10,
    fill_missing_directions=True,
    output_path="distribution_output.csv"
):
    # Load and preprocess
    df = pd.read_csv(csv_path, parse_dates=['timestamp'])
    df = df.dropna(subset=['wsp', 'wdir', 'TI'])
    df['wdir'] = df['wdir'] % 360

    wdir_edges = np.arange(0, 360 + wdir_bin_width, wdir_bin_width)
    wdir_centers = (wdir_edges[:-1] + wdir_edges[1:]) / 2
    df['wdir_bin'] = pd.cut(df['wdir'], bins=wdir_edges, labels=wdir_centers, right=False).astype(float)

    wsp_edges = np.arange(wsp_range[0], wsp_range[1] + wsp_bin_width, wsp_bin_width)
    wsp_centers = np.round((wsp_edges[:-1] + wsp_edges[1:]) / 2, 2)

    total_count = len(df)
    mean_TI = df['TI'].mean()  

    output_rows = []
    previous_fit = None

    for wdir_bin in wdir_centers:
        df_dir = df[df['wdir_bin'] == wdir_bin].copy()
        wsp_data = df_dir['wsp']
        n = len(wsp_data)
        p_dir = n / total_count

        if n >= min_samples:
            c, loc, scale = weibull_min.fit(wsp_data, floc=0)
            previous_fit = (c, scale)
        elif fill_missing_directions and previous_fit is not None:
            c, scale = previous_fit
        else:
            continue  # skip this bin

        # Weibull CDF to get bin probabilities
        cdf_vals = weibull_min(c, loc=0, scale=scale).cdf(wsp_edges)
        cond_probs = np.diff(cdf_vals)  # p(wsp | wdir)

        # Loop over wind speed bins and compute mean TI
        df_dir['wsp_bin'] = pd.cut(df_dir['wsp'], bins=wsp_edges, labels=wsp_centers, right=False).astype(float)
        ti_by_bin = df_dir.groupby('wsp_bin', observed=True)['TI'].mean()
        


        for wsp_center, p_wsp_given_dir in zip(wsp_centers, cond_probs):
            p_joint = p_dir * p_wsp_given_dir
            mean_ti = ti_by_bin.get(wsp_center, np.nan)
            if np.isnan(mean_ti):
                mean_ti = mean_TI  # fallback to site mean

            output_rows.append({
                "wsp": round(wsp_center, 2),
                "wdir": round(wdir_bin, 2),
                "TI": round(mean_ti, 4) if pd.notna(mean_ti) else np.nan,
                "probability": p_joint,
                "n": n
            })


    dist_df = pd.DataFrame(output_rows)
    operational_min = 3
    operational_max = 25
    op_mask = (dist_df['wsp'] >= operational_min) & (dist_df['wsp'] <= operational_max)
    dist_df = dist_df[op_mask].copy()
    dist_df['probability'] /= dist_df['probability'].sum()  # Renormalize


    dist_df.to_csv(output_path, index=False)
    return dist_df
output_path = "/groups/SUDOCO/Task23/sudoco_task2.3/distributions/test.csv"
result = compute_distribution(
    csv_path="/groups/SUDOCO/Task23/sudoco_task2.3/data/timeseries/HKNB_timeseries_full_filled_small_gaps_only_TI_boost5.csv",
    wdir_bin_width=3,
    wsp_bin_width=1,
    wsp_range=(0, 34),
    fill_missing_directions=True,
    output_path=output_path
)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ---- USER PARAMS ----
distribution_csv = output_path

timeseries_csv = "/groups/SUDOCO/Task23/sudoco_task2.3/data/timeseries/HKNB_timeseries.csv" # original time series file
wsp_bin_width = 1
wdir_bin_width = 5
wsp_range = (0, 34)
selected_dirs = [2.5, 92.5,182.5,272.5]  # wind direction bin centers (float)
# ----------------------

# ----------------------

# Load and preprocess time series
ts_df = pd.read_csv(timeseries_csv, parse_dates=["timestamp"])
ts_df = ts_df.dropna(subset=["wsp", "wdir"])
ts_df["wdir"] = ts_df["wdir"] % 360

# Bin definitions
wdir_edges = np.arange(0, 360 + wdir_bin_width, wdir_bin_width)
wdir_centers = (wdir_edges[:-1] + wdir_edges[1:]) / 2
wsp_edges = np.arange(wsp_range[0], wsp_range[1] + wsp_bin_width, wsp_bin_width)
wsp_centers = np.round((wsp_edges[:-1] + wsp_edges[1:]) / 2, 2)

# Assign bins
ts_df["wdir_bin"] = pd.cut(ts_df["wdir"], bins=wdir_edges, labels=wdir_centers, right=False).astype(float)
ts_df["wsp_bin"] = pd.cut(ts_df["wsp"], bins=wsp_edges, labels=wsp_centers, right=False).astype(float)

# Load fitted distribution
dist_df = pd.read_csv(distribution_csv)

# Plotly figure
fig = go.Figure()

# Plot for each direction bin
for wdir in selected_dirs:
    # Match direction safely (floating point)
    ts_mask = np.isclose(ts_df["wdir_bin"], wdir)
    if not ts_mask.any():
        continue  # skip if no data

    wsp_counts = ts_df[ts_mask]["wsp_bin"].value_counts().sort_index()
    wsp_counts = wsp_counts.reindex(wsp_centers, fill_value=0)
    prob_empirical = wsp_counts / wsp_counts.sum()

    # Fitted data
    prob_fitted = dist_df[np.isclose(dist_df["wdir"], wdir)].set_index("wsp")["probability"]
    prob_fitted = prob_fitted.reindex(wsp_centers, fill_value=0)
    prob_fitted /= prob_fitted.sum()

    fig.add_trace(go.Scatter(
        x=wsp_centers,
        y=prob_empirical,
        mode='lines+markers',
        name=f"{wdir:.1f}° empirical"
    ))
    fig.add_trace(go.Scatter(
        x=wsp_centers,
        y=prob_fitted,
        mode='lines',
        name=f"{wdir:.1f}° fitted"
    ))

# Layout
fig.update_layout(
    title="Measured vs Fitted Wind Speed Distributions for Selected Directions",
    xaxis_title="Wind Speed Bin Center [m/s]",
    yaxis_title="Probability",
    hovermode="x unified",
    template="plotly_white",
    width=900,
    height=600
)

fig.show()
